# 05b · Image Classification

Given an image, predict which class it belongs to.  
We'll build CNNs from scratch, then see why **transfer learning** is the practical default.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## What Is Image Classification?

Input: a single image.  
Output: a probability distribution over classes.

```
  🖼️ Image  →  [ CNN ]  →  "cat" (92%), "dog" (6%), "bird" (2%)
```

In [ ]:
mnist_train = torchvision.datasets.MNIST(root="./data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST(root="./data", train=False, download=True)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    idx = next(j for j, (_, l) in enumerate(mnist_train) if l == i)
    img, label = mnist_train[idx]
    axes[0, i].imshow(np.array(img), cmap="gray")
    axes[0, i].set_title(f"{label}", fontsize=12)
    axes[0, i].axis("off")

cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
for i in range(10):
    idx = next(j for j, (_, l) in enumerate(cifar_train) if l == i)
    img, label = cifar_train[idx]
    axes[1, i].imshow(np.array(img))
    axes[1, i].set_title(cifar_train.classes[label], fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("MNIST", fontsize=11)
axes[1, 0].set_ylabel("CIFAR", fontsize=11)
plt.tight_layout()
plt.show()

---
## Simple CNN on MNIST

MNIST: 28×28 grayscale digits (0–9). The "Hello World" of computer vision.

Architecture:
```
Input (1×28×28)
  → Conv(1→32, 3×3) + ReLU + MaxPool(2)
  → Conv(32→64, 3×3) + ReLU + MaxPool(2)
  → Flatten → FC(1600→128) → ReLU → FC(128→10)
```

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = MNISTNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])

train_set = torchvision.datasets.MNIST("./data", train=True, transform=transform)
test_set = torchvision.datasets.MNIST("./data", train=False, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/5 — Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc = (all_preds == all_labels).mean()
print(f"Test Accuracy: {test_acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(ax=ax, cmap="Blues")
ax.set_title(f"MNIST Confusion Matrix — {test_acc:.1%} Accuracy")
plt.tight_layout()
plt.show()

In [ ]:
wrong_mask = all_preds != all_labels
wrong_indices = np.where(wrong_mask)[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
raw_test = torchvision.datasets.MNIST("./data", train=False)
for i, ax in enumerate(axes.flat):
    if i >= len(wrong_indices):
        break
    idx = wrong_indices[i]
    img, true_label = raw_test[idx]
    ax.imshow(np.array(img), cmap="gray")
    ax.set_title(f"True: {true_label}, Pred: {all_preds[idx]}", color="red", fontsize=10)
    ax.axis("off")

fig.suptitle("Misclassified Examples", fontsize=14)
plt.tight_layout()
plt.show()

---
## CIFAR-10 with a Deeper CNN

CIFAR-10 is much harder than MNIST: color images, real-world objects, only 32×32 pixels.

We need a deeper network with batch normalization and data augmentation.

In [ ]:
class CIFAR10Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            self._block(3, 32),
            self._block(32, 64),
            self._block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 10),
        )

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cifar_model = CIFAR10Net().to(device)
print(f"Parameters: {sum(p.numel() for p in cifar_model.parameters()):,}")

In [ ]:
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

c_train = torchvision.datasets.CIFAR10("./data", train=True, transform=train_tf, download=True)
c_test = torchvision.datasets.CIFAR10("./data", train=False, transform=test_tf)

c_train_loader = DataLoader(c_train, batch_size=128, shuffle=True, num_workers=2)
c_test_loader = DataLoader(c_test, batch_size=256)

In [ ]:
optimizer = optim.Adam(cifar_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
EPOCHS = 15

for epoch in range(EPOCHS):
    cifar_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in c_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cifar_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    scheduler.step()

    history["train_loss"].append(running_loss / total)
    history["train_acc"].append(correct / total)

    cifar_model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in c_test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cifar_model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)
            test_correct += (outputs.argmax(1) == labels).sum().item()
            test_total += labels.size(0)

    history["test_loss"].append(test_loss / test_total)
    history["test_acc"].append(test_correct / test_total)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {history['train_loss'][-1]:.4f} Acc: {history['train_acc'][-1]:.3f} | "
          f"Test Loss: {history['test_loss'][-1]:.4f} Acc: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], label="Train", marker="o", markersize=4)
ax1.plot(history["test_loss"], label="Test", marker="s", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], label="Train", marker="o", markersize=4)
ax2.plot(history["test_acc"], label="Test", marker="s", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final test accuracy: {history['test_acc'][-1]:.1%}")

In [ ]:
raw_cifar_test = torchvision.datasets.CIFAR10("./data", train=False)
class_names = raw_cifar_test.classes

cifar_model.eval()
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
indices = np.random.choice(len(c_test), 16, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    img_tensor, true_label = c_test[idx]
    raw_img = np.array(raw_cifar_test[idx][0])

    with torch.no_grad():
        pred = cifar_model(img_tensor.unsqueeze(0).to(device)).argmax(1).item()

    color = "green" if pred == true_label else "red"
    ax.imshow(raw_img)
    ax.set_title(f"{class_names[pred]}", color=color, fontsize=10)
    ax.axis("off")

plt.suptitle("CIFAR-10 Predictions (green=correct, red=wrong)", fontsize=13)
plt.tight_layout()
plt.show()

---
## Transfer Learning

The **most practical technique** in computer vision.

**The idea**: take a model trained on ImageNet (1.2 million images, 1000 classes),  
replace the last layer, and fine-tune on *your* small dataset.

**Why it works**: early layers learn universal features (edges, textures, shapes)  
that transfer across tasks. Only the final classification layer is task-specific.

```
ImageNet model:  [Conv layers: edges→textures→parts→objects] → FC(1000 classes)
                                                                ↓ replace
Your model:      [Conv layers: edges→textures→parts→objects] → FC(YOUR classes)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                  These are FROZEN — already know useful features
```

### Transfer Learning in Practice

Steps:
1. Load a pretrained model (e.g., ResNet18)
2. **Freeze** the base layers (`requires_grad = False`)
3. **Replace** the final FC layer for your number of classes
4. Train only the new head (fast!)
5. Optionally: unfreeze and fine-tune everything with a tiny learning rate

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Sequential(
    nn.Linear(resnet.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
)
resnet = resnet.to(device)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
tl_train_tf = T.Compose([
    T.Resize(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
tl_test_tf = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

tl_train = torchvision.datasets.CIFAR10("./data", train=True, transform=tl_train_tf)
tl_test = torchvision.datasets.CIFAR10("./data", train=False, transform=tl_test_tf)

small_train = Subset(tl_train, range(5000))
small_test = Subset(tl_test, range(1000))

tl_train_loader = DataLoader(small_train, batch_size=64, shuffle=True)
tl_test_loader = DataLoader(small_test, batch_size=128)

print(f"Training on {len(small_train)} images, testing on {len(small_test)} images")

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, epochs, lr=1e-3):
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    results = []

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
                total += labels.size(0)
        acc = correct / total
        results.append(acc)
        print(f"  Epoch {epoch+1}/{epochs} — Test Acc: {acc:.4f}")

    return results

print("Transfer Learning (frozen base):")
tl_results = train_and_evaluate(resnet, tl_train_loader, tl_test_loader, epochs=5)

In [ ]:
scratch_model = models.resnet18(weights=None)
scratch_model.fc = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10),
)
scratch_model = scratch_model.to(device)

print("Training from SCRATCH (no pretrained weights):")
scratch_results = train_and_evaluate(scratch_model, tl_train_loader, tl_test_loader, epochs=5)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 6), tl_results, "o-", label="Transfer Learning", linewidth=2)
ax.plot(range(1, 6), scratch_results, "s--", label="From Scratch", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy")
ax.set_title("Transfer Learning vs Training from Scratch\n(Same architecture, same small dataset)")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Common Architectures — Know They Exist

You don't need to implement these — just know when to reach for each one.

| Architecture | Year | Key Innovation | When to Use |
|---|---|---|---|
| **VGG** | 2014 | Simple 3×3 conv stacks | Teaching / understanding |
| **ResNet** | 2015 | Skip connections (residuals) | Default go-to for most tasks |
| **EfficientNet** | 2019 | Compound scaling | When you need accuracy + efficiency |
| **ViT** | 2020 | Transformer on image patches | Large datasets, state-of-the-art |
| **ConvNeXt** | 2022 | CNN redesigned with ViT ideas | Modern CNN alternative to ViT |

**Practical rule of thumb**:
- Small project / learning? → **ResNet18 or ResNet50**
- Production / mobile? → **EfficientNet-B0 to B4**
- Large data + compute? → **ViT or ConvNeXt**

---
## Grad-CAM — What Is the Model Looking At?

Grad-CAM (Gradient-weighted Class Activation Mapping) highlights which regions  
of the image most influenced the model's prediction. It uses the gradients  
flowing into the last convolutional layer to produce a heatmap.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(1).item()

        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class

In [ ]:
resnet_for_cam = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
resnet_for_cam.eval()
gradcam = GradCAM(resnet_for_cam, resnet_for_cam.layer4[-1])

imagenet_labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
try:
    import urllib.request
    response = urllib.request.urlopen(imagenet_labels_url)
    imagenet_labels = [line.decode("utf-8").strip() for line in response.readlines()]
except:
    imagenet_labels = [f"class_{i}" for i in range(1000)]

raw_cifar = torchvision.datasets.CIFAR10("./data", train=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    pil_img = raw_cifar[i * 100][0]
    input_tensor = tl_test_tf(pil_img).unsqueeze(0).to(device)

    cam_map, pred_class = gradcam.generate(input_tensor)
    resized_img = pil_img.resize((224, 224))

    axes[0, i].imshow(resized_img)
    axes[0, i].set_title(f"Pred: {imagenet_labels[pred_class]}", fontsize=10)
    axes[0, i].axis("off")

    axes[1, i].imshow(resized_img)
    axes[1, i].imshow(cam_map, cmap="jet", alpha=0.5)
    axes[1, i].set_title("Grad-CAM Heatmap", fontsize=10)
    axes[1, i].axis("off")

plt.suptitle("Grad-CAM: Where the Model Looks", fontsize=14)
plt.tight_layout()
plt.show()

---

### Key Takeaways

| Concept | Remember |
|---|---|
| MNIST CNN | 2 conv layers + FC → >99% easily |
| CIFAR-10 CNN | Need BatchNorm + augmentation + more depth |
| Transfer learning | **Default approach** — pretrained features + new head |
| Freeze vs fine-tune | Freeze first (fast), then optionally unfreeze with tiny LR |
| Grad-CAM | Sanity-check what your model actually looks at |
| Architecture choice | ResNet for learning, EfficientNet for production, ViT for SOTA |